# Baseline Determinization UCT MCTS for Carcassonne

This notebook demonstrates a maintainable architecture where all simulator, agent, MCTS, and evaluation logic lives in the `carc_rl` package (`src/`) and the notebook only orchestrates experiments.

**Determinization approach:** each MCTS simulation clones the state and shuffles the remaining deck in that clone. This approximates hidden-information sampling while preserving the currently visible `next_tile`.  
**Engine mapping:** the adapter wraps the WingedSheep engine API and isolates engine-specific details in `engine_adapter.py`.

In [ ]:
%pip install -e engine
%pip install -e .

In [ ]:
from carc_rl import CarcassonneSim, RandomAgent, GreedyAgent, MCTSAgent
from carc_rl.eval import play_game, run_match
import random

In [ ]:
sim = CarcassonneSim(players=2)
state = sim.reset(seed=123)
rng = random.Random(123)
agent = RandomAgent()

print('Initial:', sim.render_text(state))
for i in range(6):
    action = agent.select_action(sim, state, rng)
    state = sim.step(state, action, rng)
    print(f'Move {i+1}:', sim.render_text(state))

In [ ]:
!pytest -q

In [ ]:
import time
from carc_rl.mcts.rollout import rollout_random

def _time_per_move(agent, sims_list, seed=0):
    rows = []
    for sims in sims_list:
        agent.n_simulations = sims
        sim = CarcassonneSim(players=2)
        state = sim.reset(seed=seed)
        rng = random.Random(seed)
        start = time.perf_counter()
        _ = agent.select_action(sim, state, rng)
        elapsed = time.perf_counter() - start
        rows.append({"sims": sims, "sec_per_move": elapsed})
    return rows

sims_list = [100, 200, 500, 1000]

baseline_agent = MCTSAgent(n_simulations=100, batch_size=1, progressive_widening=False, seed=0)
optimized_agent = MCTSAgent(n_simulations=100, batch_size=100, progressive_widening=True, seed=0)

baseline_rows = _time_per_move(baseline_agent, sims_list)
optimized_rows = _time_per_move(optimized_agent, sims_list)

summary = []
for base, opt in zip(baseline_rows, optimized_rows):
    speedup = base["sec_per_move"] / opt["sec_per_move"] if opt["sec_per_move"] else float('inf')
    summary.append({
        "sims": base["sims"],
        "baseline_sec": round(base["sec_per_move"], 4),
        "optimized_sec": round(opt["sec_per_move"], 4),
        "speedup": round(speedup, 2),
    })

summary

In [ ]:
sim = CarcassonneSim(players=2)
state = sim.reset(seed=7)
rng = random.Random(7)
actions = sim.legal_actions(state)
example_action = actions[0]

def _avg_time(fn, n=50):
    start = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - start) / n

clone_time = _avg_time(lambda: sim.clone(state))
legal_time = _avg_time(lambda: sim.legal_actions(state))
step_time = _avg_time(lambda: sim.step(sim.clone(state), example_action, rng))
rollout_time = _avg_time(lambda: rollout_random(sim, state, sim.current_player(state), rng, max_depth=30, heuristic_fn=sim.heuristic_value), n=20)

breakdown = [
    {"op": "clone", "avg_sec": round(clone_time, 6)},
    {"op": "legal_actions", "avg_sec": round(legal_time, 6)},
    {"op": "step", "avg_sec": round(step_time, 6)},
    {"op": "rollout(30)", "avg_sec": round(rollout_time, 6)},
]

breakdown

In [ ]:
sim = CarcassonneSim(players=2)
results = {}

results['Random vs Random'] = run_match(sim, RandomAgent(), RandomAgent(), n_games=3, seed=0, max_moves=80)
results['Greedy vs Random'] = run_match(sim, GreedyAgent(), RandomAgent(), n_games=3, seed=100, max_moves=80)
results['MCTS(200) vs Greedy'] = run_match(sim, MCTSAgent(n_simulations=200, seed=1), GreedyAgent(), n_games=1, seed=200, max_moves=12)
results['MCTS(1000) vs Greedy'] = run_match(sim, MCTSAgent(n_simulations=1000, seed=1), GreedyAgent(), n_games=1, seed=300, max_moves=5)

results